# K - Knowledge Transfer (Wissenstransfer)

## QUA³CK-Phase

Die K-Phase dokumentiert und kommuniziert Ergebnisse und überführt sie in
nutzbare Artefakte. Im Projekt umfasst das die reproduzierbare
Notebook-Dokumentation, gemeinsame Python-Kernlogik, Tests, README und eine
interaktive Streamlit-App.

## Umsetzung im Projekt

Die Zielgruppe kann meteorologische Szenarien einstellen, eine normierte
PV-Prognose samt empirischem 80-%-Intervall ansehen, den thermischen Effekt bei
konstanter Einstrahlung untersuchen und das Szenario mit besonders
ertragreichen beobachteten Stunden vergleichen.

### Live-Anwendung

[PV Weather Predictor Germany auf Streamlit öffnen](https://bigdataphotovoltaikerzeugung-eavpwvxvq67ts3b2ebjpnz.streamlit.app/)


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

pd.set_option("display.max_columns", 30)


In [ ]:
from pv_weather import (
    estimate_module_temperature,
    load_project_data,
    predict_yield,
    train_yield_model,
)

data, source = load_project_data(
    ROOT / "data" / "processed" / "hourly_pv_weather.csv"
)
bundle = train_yield_model(data)
print(source)


## Zielgruppengerechte Projektzusammenfassung


In [ ]:
summary = pd.Series(
    {
        "Forschungsfrage": (
            "Unter welchen meteorologischen Bedingungen ist die normierte "
            "PV-Erzeugung in Deutschland am höchsten?"
        ),
        "Daten": "Stündliche SMARD-PV-Daten und DWD-Wetterdaten",
        "Zielvariable": "PV-Erzeugung / installierte PV-Leistung / 1 h",
        "Modell": "HistGradientBoosting mit meteorologischen Merkmalen",
        "Validierung": f"Zeitlicher Test ab {bundle.split_timestamp:%d.%m.%Y}",
        "Modell-MAE": f"{bundle.metrics['model_mae'] * 100:.2f} Prozentpunkte",
        "Baseline-MAE": f"{bundle.metrics['baseline_mae'] * 100:.2f} Prozentpunkte",
        "Anwendung": "Interaktive Streamlit-App",
    },
    name="PV-Wetter-Projekt",
)
display(summary.to_frame())


## Beispiel für den Transfer vom Modell zur Anwendung


In [ ]:
scenario = pd.DataFrame(
    {
        "timestamp_utc": [pd.Timestamp("2024-01-01", tz="UTC")],
        "temperature_c": [25.0],
        "relative_humidity_pct": [55.0],
        "global_radiation_j_cm2": [270.0],
        "diffuse_radiation_j_cm2": [78.3],
        "sunshine_duration_min": [38.2],
        "cloud_cover_oktas": [2.0],
        "wind_speed_m_s": [3.0],
    }
)
prediction = predict_yield(bundle, scenario).iloc[0]
module_temperature = estimate_module_temperature(
    scenario["temperature_c"],
    scenario["global_radiation_j_cm2"],
    scenario["wind_speed_m_s"],
)[0]

result = pd.Series(
    {
        "Normierte Prognose (%)": prediction["normalized_pv_prediction"] * 100,
        "Untere 80-%-Grenze (%)": prediction["lower_80"] * 100,
        "Obere 80-%-Grenze (%)": prediction["upper_80"] * 100,
        "Geschätzte Modultemperatur (°C)": module_temperature,
    },
    name="Beispielszenario",
)
display(result.round(2).to_frame())


Die installierte PV-Leistung ist kein Modellmerkmal. Sie skaliert die normierte
Prognose nachträglich zu MW beziehungsweise MWh pro Stunde. Dadurch bleibt die
meteorologische Aussage unabhängig vom Ausbaugrad.


## Artefakt- und Kommunikationslandkarte


In [ ]:
artifacts = pd.DataFrame(
    [
        ("app.py", "Interaktive Prognose und Ergebnisvermittlung", "Zielgruppe"),
        ("README.md", "Projektüberblick, Installation und Grenzen", "Portfolio / Entwicklung"),
        ("notebooks/", "Nachvollziehbare QUA³CK-Dokumentation", "Lehre / Prüfung"),
        ("pv_weather/", "Wiederverwendbare Kernlogik", "Entwicklung"),
        ("tests/", "Automatisierte Qualitätsverträge", "Entwicklung / Wartung"),
        ("data/README.md", "Datenquellen, Schema und Aufbereitung", "Reproduzierbarkeit"),
        ("KI_NUTZUNG.md", "Transparenz zur KI-Unterstützung", "Prüfung / Portfolio"),
    ],
    columns=["Artefakt", "Transferfunktion", "Adressaten"],
)
artifacts["vorhanden"] = artifacts["Artefakt"].map(lambda path: (ROOT / path).exists())
display(artifacts)


## Reproduzierbare Nutzung

Im Projektverzeichnis:

```powershell
# Abhängigkeiten
python -m venv .venv
.venv\Scripts\Activate.ps1
python -m pip install -r requirements.txt

# Realdaten laden und Modell prüfen
python scripts/download_real_data.py --start-year 2020 --end-year 2025

# QUA³CK-Notebooks neu erzeugen und ausführen
python scripts/create_quack_notebooks.py --execute

# Tests
python -m pytest

# Anwendung
streamlit run app.py
```

Ohne lokale Rohdaten nutzt die Anwendung klar markierte synthetische
Demodaten. Diese demonstrieren den Workflow, sind aber kein empirischer Befund.


## Checkliste für Wissenstransfer und Bereitstellung


In [ ]:
transfer_checklist = pd.DataFrame(
    [
        ("Forschungsfrage und Zielgruppe dokumentiert", True),
        ("Datenquelle und Demo-Status sichtbar", True),
        ("Modell gegen Baseline validiert", bundle.metrics["model_mae"] < bundle.metrics["baseline_mae"]),
        ("Unsicherheitsintervall kommuniziert", True),
        ("Grenzen und fehlende Einflussgrößen dokumentiert", True),
        ("Streamlit-App vorhanden", (ROOT / "app.py").exists()),
        ("Automatisierte Tests vorhanden", (ROOT / "tests").exists()),
        ("Reproduzierbare Notebook-Erzeugung vorhanden", (ROOT / "scripts" / "create_quack_notebooks.py").exists()),
    ],
    columns=["Kriterium", "erfüllt"],
)
display(transfer_checklist)
assert transfer_checklist["erfüllt"].all(), "Die Wissenstransfer-Checkliste ist unvollständig."


## Kernaussage und verantwortungsvolle Kommunikation

Hohe Einstrahlung ist der wichtigste beobachtete Treiber hoher normierter
PV-Erzeugung. Temperaturunterschiede müssen bei vergleichbarer Einstrahlung
bewertet werden. Beobachtete Gruppenunterschiede und Modellreaktionen sind
keine Kausalnachweise und keine Ertragsgarantie.

Für einen produktiven Dauerbetrieb wären zusätzlich Monitoring von
Datenqualität und Model Drift, Modellversionierung, wiederholbares
Experiment-Tracking und eine dokumentierte Freigabe neuer Modellversionen
erforderlich.

## Technische Verankerung der K-Phase

- `app.py`: interaktive Bereitstellung
- `README.md`: Portfolio- und Nutzungsdokumentation
- `scripts/create_quack_notebooks.py`: reproduzierbare Phasendokumentation
- `pv_weather/workflow.py`: kontrollierter Daten- und Trainingsablauf
- `tests/`: Qualitätskontrolle
- `KI_NUTZUNG.md`: Transparenz
